# NITS-IQA analysis (English publication adaptation)

This notebook is adapted from a historical experiment notebook for an English-only repository. The original notebook remains in the local archive. Source data are the 405 recorded pairs; no source images or model inference are included. Numeric calculations retain the historical logic, with an equivalent binary balanced-accuracy implementation and explicit repairs of invalid or out-of-order cells.

In [ ]:
from pathlib import Path
import pandas as pd

def locate_nits_csv():
    """Find the English-header NITS-IQA scores relative to the notebook or repository."""
    base = Path.cwd().resolve()
    candidates = [base / "nits_iqa/verified/NITS_IQA_405_verified.csv",
                  base / "../verified/NITS_IQA_405_verified.csv",
                  base / "verified/NITS_IQA_405_verified.csv"]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError("NITS_IQA_405_verified.csv not found; run from repository root or notebook directory")

DATA_CSV = locate_nits_csv()

def fast_balanced_accuracy(y_true, y_pred):
    """Binary balanced accuracy, equivalent to sklearn on these two-class folds."""
    import numpy as np
    yt = np.asarray(y_true)
    yp = np.asarray(y_pred)
    pos = yt == 1
    neg = yt == 0
    if not pos.any() or not neg.any():
        raise ValueError("Both classes are required for this binary benchmark")
    return 0.5 * ((yp[pos] == 1).mean() + (yp[neg] == 0).mean())


In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    balanced_accuracy_score
)

from sklearn.model_selection import LeaveOneGroupOut

file = DATA_CSV
df = pd.read_csv(file)

metrics = [
    "DINOv2-Base (%)",
    "DINOv2-Large (%)",
    "Dists (%)",
    "SSIM (%)",
    "OpenCLIP (%)"
]

y = (df["MOS"] >= 50).astype(int).to_numpy()
groups = df["Ref"].to_numpy()

logo = LeaveOneGroupOut()
results = []

for metric in metrics:

    scores = df[metric].to_numpy()
    predictions = np.zeros(len(df), dtype=int)

    for train_idx, test_idx in logo.split(scores, y, groups):

        train_scores = scores[train_idx]
        train_labels = y[train_idx]

        # Select the threshold using training data only
        thresholds = np.unique(train_scores)
        best_threshold = thresholds[0]
        best_score = -1

        for threshold in thresholds:
            pred = (train_scores >= threshold).astype(int)
            score = fast_balanced_accuracy(train_labels, pred)

            if score > best_score:
                best_score = score
                best_threshold = threshold

        predictions[test_idx] = (
            scores[test_idx] >= best_threshold
        ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y, predictions, labels=[0, 1]
    ).ravel()

    results.append({
        "Metric": metric,
        "Accuracy": accuracy_score(y, predictions),
        "Precision": precision_score(y, predictions),
        "Recall": recall_score(y, predictions),
        "F1": f1_score(y, predictions),
        "FPR": fp / (fp + tn),
        "FNR": fn / (fn + tp)
    })

result_df = pd.DataFrame(results)

print(result_df.round(4).to_string(index=False))